# 01 - AI Agent Foundations: The Architecture Ladder

## Scenario: Deciding how to automate SaaS support

Before you build a complex multi-agent system, you must ask: **Does this task actually need an agent?** 

A core principle of AI Engineering is the **Architecture Ladder**: always use the least-autonomous architecture that reliably solves the problem. In this notebook, we will solve Northstar's support ticket triage problem by walking up the ladder, from basic automation to a bounded agent. We will use `openai`, `pydantic`, and `chromadb` to implement these patterns.

In [1]:
import os
import json
from openai import OpenAI

# Initialize OpenAI Client
# 1. Attempt to use the real API
if os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
else:
    # 2. Fallback to our local mock for students without keys
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    import sys
    import os
    sys.path.append(os.path.abspath("../../.."))
    from awsome_agents.mock_openai import MockOpenAI
    client = MockOpenAI()

# 3. Optional: Local LLMs
# If you prefer to use a local model like Llama 3 instead of the mock:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Sample incoming support tickets
tickets = [
    "I need to reset my password.",
    "The EU checkout page is throwing a 500 error when I use a credit card.",
    "What is your refund policy for annual plans?"
]


## Level 1: Traditional Automation (No AI)

If a path is known and requires no complex reasoning, fixed code is safer, faster, and infinitely cheaper than an LLM.

**Use case**: Simple intent routing based on keywords.

In [2]:
def traditional_router(ticket: str) -> str:
    ticket_lower = ticket.lower()
    if "password" in ticket_lower or "login" in ticket_lower:
        return "Category: Auth"
    elif "500" in ticket_lower or "error" in ticket_lower:
        return "Category: Engineering"
    else:
        return "Category: General Support"

for t in tickets:
    print(f"Ticket: '{t[:40]}...' -> {traditional_router(t)}")


Ticket: 'I need to reset my password....' -> Category: Auth
Ticket: 'The EU checkout page is throwing a 500 e...' -> Category: Engineering
Ticket: 'What is your refund policy for annual pl...' -> Category: General Support


**Pros**: 0ms latency, free, 100% predictable.
**Cons**: Very brittle. If a user says "I can't access my account", the keyword router fails.

## Level 2: Deterministic Workflow (LLM as a Classifier)

When the input is messy but the required output is structured, we use an LLM to parse the data, but we **do not let the LLM decide what to do next**. We use `pydantic` and OpenAI's Structured Outputs to guarantee the format.

**Use case**: Robust intent routing.

In [3]:
from pydantic import BaseModel, Field

# Define our strict output schema
class TicketIntent(BaseModel):
    category: str = Field(description="One of: Auth, Billing, Engineering, General")
    urgency: int = Field(description="1 (Low) to 5 (Critical)")
    requires_human: bool = Field(description="True if the user is angry or threatening churn")

def llm_router(ticket: str) -> TicketIntent:
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-2024-08-06",
            messages=[
                {"role": "system", "content": "You are an expert triage assistant."},
                {"role": "user", "content": ticket}
            ],
            response_format=TicketIntent,
        )
        return completion.choices[0].message.parsed
    except Exception as e:
        # Mock fallback for missing API key
        return TicketIntent(category="Engineering", urgency=5, requires_human=False)

for t in tickets:
    intent = llm_router(t)
    print(f"Ticket: '{t[:40]}...' -> Category: {intent.category}, Urgency: {intent.urgency}")


Ticket: 'I need to reset my password....' -> Category: Engineering, Urgency: 5
Ticket: 'The EU checkout page is throwing a 500 e...' -> Category: Engineering, Urgency: 5
Ticket: 'What is your refund policy for annual pl...' -> Category: Engineering, Urgency: 5


**Pros**: Handles natural language nuances perfectly. Output is strictly typed.
**Cons**: Slower and more expensive than regex. Cannot answer questions that require external knowledge.

## Level 3: RAG Assistant (Retrieval-Augmented Generation)

When the task requires *current evidence* or *domain knowledge* (like answering "What is your refund policy?"), classification isn't enough. We use a vector database (`chromadb`) to inject knowledge into the prompt. The LLM synthesizes an answer, but it **still cannot take actions**.

**Use case**: Answering policy questions.

In [4]:
import chromadb

# Initialize local in-memory ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="policies")

# Add some dummy documents
collection.add(
    documents=[
        "Annual plans can be refunded within 30 days of purchase.",
        "Monthly plans are strictly non-refundable.",
        "To reset a password, use the /forgot-password endpoint."
    ],
    ids=["doc1", "doc2", "doc3"]
)

def rag_answerer(question: str) -> str:
    # 1. Retrieve relevant evidence
    results = collection.query(query_texts=[question], n_results=1)
    evidence = results['documents'][0][0]
    
    # 2. Synthesize answer using evidence
    prompt = f"Answer the user based ONLY on this policy: {evidence}\n\nUser: {question}"
    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"[Mock] Based on policy: {evidence}"

print(rag_answerer(tickets[2]))


/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 0.00/79.3M [00:00<?, ?iB/s]

/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 68.0k/79.3M [00:00<02:19, 596kiB/s]

/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 391k/79.3M [00:00<00:52, 1.57MiB/s]

/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   1%|          | 867k/79.3M [00:00<00:29, 2.77MiB/s]

/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   2%|▏         | 1.61M/79.3M [00:00<00:17, 4.56MiB/s]

...

/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  97%|█████████▋| 77.1M/79.3M [00:11<00:00, 7.69MiB/s]

/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  98%|█████████▊| 77.9M/79.3M [00:11<00:00, 7.79MiB/s]

/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  99%|█████████▉| 78.7M/79.3M [00:12<00:00, 7.08MiB/s]

/Users/mahsateimourikia/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:12<00:00, 6.83MiB/s]

[Mock] Based on policy: Annual plans can be refunded within 30 days of purchase.


**Pros**: Eliminates hallucinations. Answers complex questions reliably.
**Cons**: Cannot run diagnostics. Cannot restart servers. It is purely read-only.

## Level 4: Bounded Agent (Tools + Reasoning Loop)

When the task requires *dynamic tool choice* (e.g., investigating an engineering issue), we upgrade to an Agent. The agent is given tools and a loop to observe, decide, and act.

**Use case**: Investigating the EU checkout error.

In [5]:
# Define a tool for the agent
tools = [{
    "type": "function",
    "function": {
        "name": "check_server_health",
        "description": "Check the health of a regional server.",
        "parameters": {
            "type": "object",
            "properties": {"region": {"type": "string"}},
            "required": ["region"]
        }
    }
}]

def check_server_health(region: str) -> str:
    print(f"[Tool Executed] Checking health for {region}...")
    return '{"cpu": "99%", "status": "failing"}'

def agent_investigator(ticket: str):
    messages = [
        {"role": "system", "content": "You are a DevOps agent. Use tools to investigate."},
        {"role": "user", "content": ticket}
    ]
    
    try:
        # Step 1: Model decides to use a tool
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools
        )
        
        tool_call = response.choices[0].message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        
        # Step 2: Application executes the tool
        result = check_server_health(**args)
        
        # Step 3: Model synthesizes final answer
        messages.append(response.choices[0].message)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "name": tool_call.function.name, "content": result})
        
        final_response = client.chat.completions.create(model="gpt-4o", messages=messages)
        print("\nAgent Conclusion:", final_response.choices[0].message.content)
    except Exception as e:
        print("Agent Mock Execution: Detected EU issue, checked server, CPU is 99%.")

agent_investigator(tickets[1])


Agent Mock Execution: Detected EU issue, checked server, CPU is 99%.


**Pros**: Autonomous problem-solving. Can string multiple tools together.
**Cons**: Unpredictable. Can get stuck in infinite loops. Hard to debug.

## Level 5: Human-Approved Workflow

Just because an agent *can* solve a problem doesn't mean it has the *authority* to fix it. For irreversible actions (like restarting a server or issuing a refund), we combine the Agent's reasoning with a Human approval step.

**Use case**: Proposing a server restart instead of executing it directly.

In [6]:
class RestartProposal(BaseModel):
    region: str
    reason: str
    downtime_estimate_mins: int

def propose_restart(proposal: RestartProposal):
    print(f"\n🚨 [HUMAN APPROVAL REQUIRED] 🚨")
    print(f"Agent wants to restart {proposal.region}.")
    print(f"Reason: {proposal.reason}")
    print("Action paused in queue...")

# Using Pydantic to ensure the agent formats its dangerous request perfectly
# propose_restart(RestartProposal(region="eu-west", reason="CPU at 99%", downtime_estimate_mins=5))


## Watch For

- **Skipping the Ladder**: Don't build an Agent when a traditional regex router or RAG system is sufficient. Agents are slow, expensive, and non-deterministic.
- **Authority vs Reasoning**: An LLM's ability to reason about a problem does not grant it the authority to take irreversible actions. Always fall back to Human-Approved Workflows for dangerous tools.

## Checkpoint

**1. What is the main difference between a RAG Assistant (Level 3) and an Agent (Level 4)?**
- A) RAG uses Vector DBs; Agents do not.
- B) RAG only reads data and generates text; Agents can dynamically choose and execute tools to alter their environment.
- C) Agents are always faster than RAG.
- D) RAG cannot use OpenAI.

**2. Which of the following tasks should immediately be escalated to a Human-Approved Workflow?**
- A) Summarizing a long support ticket.
- B) Querying a customer's order history.
- C) Processing a $500 refund to a user's credit card.
- D) Translating an email from French to English.
